In [28]:

import pymupdf4llm
import pathlib 
from langchain.document_loaders import PyMuPDFLoader
def convert_pdf_to_markdown(pdf_name, mk_name):
    md_text = pymupdf4llm.to_markdown(pdf_name)
    pathlib.Path(mk_name).write_bytes(md_text.encode())
    
loader = PyMuPDFLoader('test2.pdf')
docs = loader.load()


In [29]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter, SentenceTransformersTokenTextSplitter, SpacyTextSplitter

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,
#     chunk_overlap=200,
#     length_function=len
# )
text_splitter_2 = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)
# text_splitter_3 = SentenceTransformersTokenTextSplitter(
#     model_name="all-MiniLM-L6-v2",
#     chunk_size=1000,
#     chunk_overlap=200,
#     length_function=len
# )
# text_splitter_4 = SpacyTextSplitter(
#     pipeline="en_core_web_sm",
#     chunk_size=1000,
#     chunk_overlap=200,
#     length_function=len
# )

def split_docs(docs, text_splitter):
    """Splits documents into smaller chunks."""
    # Split the documents into smaller chunks
    return text_splitter.split_documents(docs)

splitted_docs = split_docs(docs, text_splitter_2)

splitted_docs

[Document(metadata={'producer': 'Aspose.Pdf for .NET 17.6', 'creator': 'Aspose Ltd.', 'creationdate': '', 'source': 'test2.pdf', 'file_path': 'test2.pdf', 'total_pages': 13, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-01-27T15:06:29-05:00', 'trapped': '', 'modDate': "D:20200127150629-05'00'", 'creationDate': '', 'page': 0}, page_content='ODM - SUPPLY AGREEMENT\nBETWEEN:\nORGANIC PREPARATIONS INC.\n2nd Floor, Transpacific Haus\nLini Highway, Port Vila. Vanuatu\n“the Manufacturer”\n-- AND --\nAGAPE ATP INTERNATIONAL HOLDING LIMITED\nUnit 05, 4F, Energy Plaza\nNo. 92, Granville Road\nTsim Sha Tsui East\nKowloon, Hong Kong\n“the Customer”\nSource: AGAPE ATP CORP, 10-K/A, 12/2/2019'),
 Document(metadata={'producer': 'Aspose.Pdf for .NET 17.6', 'creator': 'Aspose Ltd.', 'creationdate': '', 'source': 'test2.pdf', 'file_path': 'test2.pdf', 'total_pages': 13, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'modda

In [ ]:
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
import sys, os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)


from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"}
)
persist_directory = "db"

vectordb = Chroma.from_documents(
    splitted_docs,
    collection_name='test',
    embedding=embeddings,
    # persist_directory=persist_directory
)
vectordb.delete_collection()
vectordb = Chroma.from_documents(
    splitted_docs,
    collection_name='test',
    embedding=embeddings
)
    # persist_directory=persist_directory
# vectordb.persist()



def search_tool(query, k=3):
    """Searches the vector store for relevant documents."""
    retriever = vectordb.as_retriever(search_kwargs={"k": k})
    return retriever.get_relevant_documents(query)



In [42]:
from Langchain.GeminiLoader import llm

In [46]:
# initialize the Langchain Agent
from langchain.agents import initialize_agent, Tool
from langchain.agents import AgentType
import json
from Langchain.GeminiLoader import llm
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field
class ResponseFormatter(BaseModel):
    """Always use this tool to structure your response to the user."""
    buyer: str = Field(default=None, description="The buyer from the response")
    seller: str = Field(default=None, description="The seller from the response")
    date: str = Field(default=None, description="The date from the response")
    contract_name: str = Field(default=None, description="The contract name from the response")
    contract_type: str = Field(default=None, description="The contract type from the response")
    # Example usage:
    # json_output = format_response_as_json(response)
    # print(json_output)
tools = [
        Tool(
            name="Search Vector Store",
            func=search_tool,
            description="useful for when you need to find something within a contract document using vector store, relevant search terms should be given. And K is the number of relevant documents to be retrieved. The output will be a list of documents with their page numbers and content. The input should be a string, and the output will be a list of strings.",
        )
        
    ]
prompt = PromptTemplate(
    input_variables=["input", "tool_names"],
    template="""You are a contract analysis agent. 
    You will be given a query and you will search the vector store for relevant documents. The query is: {input}. 
    Steps:
    1. Construct a query to search vectorstore using search_tool.
    2. Once, you retrieve the info from query, you will find the following[parties_involved, date_of_contract, contract_name, contract_type], if you don't find the info, just say you don't find it.
        You can repeat the Thought, Action, Action Input, and Observation steps up to 3 times to gather the necessary information.
    3. Once you find the answers, Format the extracted information into a proper JSON format. Ensure the JSON structure adheres to the following schema:
    {{
        "parties_involved": ["Name of all parties involved"],
        "date": "Date of the contract",
        "contract_name": "Name of the contract",
        "contract_type": "Type of the contract"
    }}"""
)
agent = initialize_agent(
    prompt=prompt,
    tools=tools,
    llm=llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    structured_response=True,
    max_iterations=3
)

# Give instructions to the agent

# Run the agent with the query
query = "Find the name of the buyer and seller, date, name and contract."

resp = agent.run(prompt.invoke({'input': query, 'tool_names': [tool.name for tool in tools]}))

resp



> Entering new AgentExecutor chain...
Okay, I understand. I need to find the parties involved, date, contract name, and contract type by searching the vector store. I will start by constructing a query to find this information.
Action: Search Vector Store
Action Input: "buyer seller date contract name"
Observation: [Document(metadata={'keywords': '', 'modDate': "D:20200127150629-05'00'", 'format': 'PDF 1.4', 'moddate': '2020-01-27T15:06:29-05:00', 'creationDate': '', 'subject': '', 'source': 'test2.pdf', 'creationdate': '', 'creator': 'Aspose Ltd.', 'author': '', 'producer': 'Aspose.Pdf for .NET 17.6', 'title': '', 'trapped': '', 'page': 7, 'file_path': 'test2.pdf', 'total_pages': 13}, page_content='12.\nTRANSFER OF INTELLECTUAL PROPERTY\nThe Manufacturer agrees to offer the Customer the first right of refusal to purchase the intellectual property for the products listed in\nSchedule A of this agreement based upon agreed terms.\n13.\nAPPOINTMENT AND GRANT OF LICENSE\n13.1\nThe Manufa

'```json\n{\n    "parties_involved": ["Organic Preparations INC.", "Agape ATP International Holding Limited"],\n    "date": "January 15, 2018",\n    "contract_name": "ODM Supply Agreement",\n    "contract_type": "Supply Agreement"\n}\n```'

In [ ]:
import json, re
doc_obj = None
if 'output' in resp:
    doc_obj = resp['output']
elif 'json' in resp:
    decoded = bytes(resp, "utf-8").decode("unicode_escape")
    # Step 2: Remove markdown code block syntax
    cleaned = re.sub(r'```json|```', '', decoded).strip().strip('"')

    # Step 3: Load JSON
    doc_obj = json.loads(cleaned)



{'parties_involved': ['Organic Preparations INC.',
  'Agape ATP International Holding Limited'],
 'date': 'January 15, 2018',
 'contract_name': 'ODM Supply Agreement',
 'contract_type': 'Supply Agreement'}

In [63]:
import spacy

nlp = spacy.load("en_core_web_lg")
ner = []
# Process spacy to get names, nouns, dates, and organizations
entities = {
    "names": [],
    "dates": [],
    "organizations": [],
    "numerical_values": []
}
for doc in splitted_docs:
    doc_analysis = nlp(doc.page_content)
    entities["names"] += [ent.text for ent in doc_analysis.ents if ent.label_ == "PERSON" and ent.text not in entities["names"]]
    entities["dates"] += [ent.text for ent in doc_analysis.ents if ent.label_ == "DATE" and ent.text not in entities["dates"]]
    entities["organizations"] += [ent.text for ent in doc_analysis.ents if ent.label_ == "ORG" and ent.text not in entities["organizations"]]
    entities["numerical_values"] += [token.text for token in doc_analysis if token.pos_ == "NUM" and token.text not in entities["numerical_values"]]

entities

{'names': ['Transpacific Haus',
  'Lily Tomas\n2',
  'Bernd Friedlander',
  'Markus Eistert',
  'Dr Ed Smith',
  'Vic Cherikoff',
  'Pavel Yutsis',
  'Michael Tirant',
  'Frank Ellis',
  'Peter Davids',
  'Rutledge Taylor',
  'Frank D.P. Ellis',
  'Michael Tait',
  'Mercy Saula\nAddress',
  'Kok Choong',
  'Ku Suat Hong',
  'Wisma Laxton',
  'Jalan Desa'],
 'dates': ['the 15th day of January 2018',
  'ten (10) years',
  'the end of every ten (10) year',
  '10) year',
  'months',
  'seven (7) days',
  'seven (7) days',
  '21 days',
  '21) days',
  'thirty consecutive days',
  'annual',
  '2000',
  'Schedule B.',
  '1) day',
  'the same day',
  'each quarter',
  'Date 15',
  '2018',
  '2018',
  '2018',
  '2018'],
 'organizations': ['ODM - SUPPLY',
  '2nd Floor',
  'Manufacturer',
  'AGAPE ATP INTERNATIONAL HOLDING LIMITED\nUnit',
  'AGAPE ATP CORP',
  'ODM',
  'RECITALS\na.',
  'Customer',
  'b.\nThe Manufacturer',
  'Customer',
  '& Agape ATP International Holding Limited',
  'Manufactu